In [25]:
import numpy as np
from time import perf_counter
from scipy.sparse.linalg import spsolve_triangular
from scipy.sparse import csc_matrix, tril, coo_matrix
from sksparse.cholmod import cholesky

In [23]:
PATH_MATRIX = "data/A0032.txt"
PATH_COLUMN = "data/rhs0032.txt"
rows, cols, vals = [], [], []

with open(PATH_MATRIX, "r") as file:
    for line in file:
        str_rows, str_cols, str_vals = line.split(' ')
        rows.append(int(str_rows))
        cols.append(int(str_cols))
        vals.append(float(str_vals))

l = max(rows) + 1
A_coo = coo_matrix((vals, (rows, cols)), shape=(l, l))
A_csc = csc_matrix(A_coo)

b = np.loadtxt(PATH_COLUMN)

if (A_csc.shape[0] != len(b)):
    print("Dimension mismatch...")


In [26]:
def my_cholesky(A_csc):
    factor = cholesky(A_csc, order="natural")
    return csc_matrix(factor[0])

t1 = perf_counter()

# Cholesky factorization needs positive definite, thus the minu
R = my_cholesky(-A_csc)
L = R.T.tocsc()

# Compensating the previous minus sign on b
y = spsolve_triangular(L, -b, lower=True)
x = spsolve_triangular(L.T, y, lower=False)

t2 = perf_counter()
tf = t2 - t1
print(f"{tf:.4f}")
print(L.nnz)


0.0130
32799
